In [22]:
from pathlib import Path

CASE_DIR = Path(r"C:\Users\misaz\lab1\case_08")

print("Папка існує:", CASE_DIR.exists())
print("\nФайли в папці:")

for file in CASE_DIR.iterdir():
    print(file.name)

Папка існує: True

Файли в папці:
.ipynb_checkpoints
alibi.json
case.json
lab1_case_08.ipynb
lab1_case_08.ipynb.ipynb
README.txt
statements.json
suspects.csv
visits_backup.csv
visits_evening.csv
visits_morning.csv


In [23]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

# Шлях до матеріалів справи
CASE_DIR = Path(r"C:\Users\misaz\lab1\case_08")

# Створюємо базу даних SQLite
engine = create_engine(
    "sqlite:///C:/Users/misaz/lab1/case_08.db"
)

# Завантажуємо чотири CSV-файли
suspects = pd.read_csv(CASE_DIR / "suspects.csv")
visits_morning = pd.read_csv(CASE_DIR / "visits_morning.csv")
visits_evening = pd.read_csv(CASE_DIR / "visits_evening.csv")
visits_backup = pd.read_csv(CASE_DIR / "visits_backup.csv")

# Зберігаємо таблиці в базі даних
suspects.to_sql("suspects", engine, if_exists="replace", index=False)
visits_morning.to_sql("visits_morning", engine, if_exists="replace", index=False)
visits_evening.to_sql("visits_evening", engine, if_exists="replace", index=False)
visits_backup.to_sql("visits_backup", engine, if_exists="replace", index=False)

print("Дані успішно завантажено до SQLite!")

print("\nПідозрювані:")
display(suspects.head())

print("\nРанкова зміна:")
display(visits_morning.head())

print("\nВечірня зміна:")
display(visits_evening.head())

print("\nРезервний журнал:")
display(visits_backup.head())

Дані успішно завантажено до SQLite!

Підозрювані:


,suspect_id,name,role
0,S01,Ігор Пасічник,реставратор
1,S02,Захар Тесля,екскурсовод
2,S03,Галина Мірошник,електрик
3,S04,Марія Снігур,охоронець
4,S05,Ярина Кушнір,науковий співробітник



Ранкова зміна:


,visit_id,suspect_id,room,hour,sensor_id
0,V001,S05,Зброярня,9,SEN-536
1,V005,S10,Зброярня,10,SEN-775
2,V007,S06,Гобеленова,12,SEN-142
3,V008,S02,Вітражна,9,SEN-650
4,V011,S07,Єгипетська,12,SEN-639



Вечірня зміна:


,visit_id,suspect_id,room,hour,sensor_id
0,V002,S02,Бароко,15,SEN-518
1,V003,S04,NaN,18,SEN-457
2,V004,S05,Гобеленова,17,SEN-646
3,V006,S05,Антична,15,SEN-351
4,V009,S01,Єгипетська,20,SEN-626



Резервний журнал:


,visit_id,room
0,V003,Жовта
1,V013,Вітражна
2,V027,Вітражна
3,V041,Дзеркальна
4,V900,Бароко


In [24]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

try:
    print("MongoDB підключено!")
    print("Версія MongoDB:", client.server_info()["version"])
except Exception as e:
    print("Помилка підключення:", e)

MongoDB підключено!
Версія MongoDB: 8.3.13


In [25]:
import json
from pathlib import Path
from pymongo import MongoClient

# Шлях до файлів вашого варіанта
CASE_DIR = Path(r"C:\Users\misaz\lab1\case_08")

# Підключаємося до MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["museum_case_08"]

# Завантажуємо свідчення та дані про алібі
with open(CASE_DIR / "statements.json", encoding="utf-8") as file:
    statements_data = json.load(file)

with open(CASE_DIR / "alibi.json", encoding="utf-8") as file:
    alibi_data = json.load(file)

# Очищаємо колекції перед завантаженням
db["statements"].drop()
db["alibi"].drop()

# Завантажуємо документи
if statements_data:
    db["statements"].insert_many(statements_data)

if alibi_data:
    db["alibi"].insert_many(alibi_data)

print("Завантажено свідчень:", db["statements"].count_documents({}))
print("Завантажено записів про алібі:", db["alibi"].count_documents({}))

Завантажено свідчень: 11
Завантажено записів про алібі: 6


In [26]:
import pandas as pd
from sqlalchemy import create_engine

# Підключаємося до SQLite
engine = create_engine("sqlite:///C:/Users/misaz/lab1/case_08.db")

# Читаємо таблиці з SQLite
suspects_sql = pd.read_sql_query("SELECT * FROM suspects", engine)
morning_sql = pd.read_sql_query("SELECT * FROM visits_morning", engine)
evening_sql = pd.read_sql_query("SELECT * FROM visits_evening", engine)
backup_sql = pd.read_sql_query("SELECT * FROM visits_backup", engine)

print("Підозрювані з SQLite:")
display(suspects_sql.head())

print("Ранкова зміна з SQLite:")
display(morning_sql.head())

print("Вечірня зміна з SQLite:")
display(evening_sql.head())

print("Резервний журнал із SQLite:")
display(backup_sql.head())

# Читаємо документи з MongoDB, виключаючи службове поле _id
statements = pd.DataFrame(
    list(db["statements"].find({}, {"_id": 0}))
)

alibi = pd.DataFrame(
    list(db["alibi"].find({}, {"_id": 0}))
)

print("Свідчення з MongoDB:")
display(statements)

print("Алібі з MongoDB:")
display(alibi)

Підозрювані з SQLite:


,suspect_id,name,role
0,S01,Ігор Пасічник,реставратор
1,S02,Захар Тесля,екскурсовод
2,S03,Галина Мірошник,електрик
3,S04,Марія Снігур,охоронець
4,S05,Ярина Кушнір,науковий співробітник


Ранкова зміна з SQLite:


,visit_id,suspect_id,room,hour,sensor_id
0,V001,S05,Зброярня,9,SEN-536
1,V005,S10,Зброярня,10,SEN-775
2,V007,S06,Гобеленова,12,SEN-142
3,V008,S02,Вітражна,9,SEN-650
4,V011,S07,Єгипетська,12,SEN-639


Вечірня зміна з SQLite:


,visit_id,suspect_id,room,hour,sensor_id
0,V002,S02,Бароко,15,SEN-518
1,V003,S04,NaN,18,SEN-457
2,V004,S05,Гобеленова,17,SEN-646
3,V006,S05,Антична,15,SEN-351
4,V009,S01,Єгипетська,20,SEN-626


... (2 outputs left out)

Свідчення з MongoDB:


,witness,saw_suspect_id,room,hour
0,вахтер Петренко,S02,Жовта,9
1,вахтер Петренко,S04,Гобеленова,13
2,волонтер Сидір,S04,Вітражна,20
3,водій Юрченко,S06,Жовта,19
4,вахтер Петренко,S08,Єгипетська,12
5,турист Новак,S10,Антична,10
6,водій Юрченко,S10,Антична,10
7,турист Новак,X01,Антична,17
8,студентка Гнатюк,X02,Бароко,17
9,волонтер Сидір,X03,Жовта,14


Алібі з MongoDB:


,suspect_id,confirmed_by
0,S01,студентка Гнатюк
1,S03,касирка Олійник
2,S04,студентка Гнатюк
3,S05,вахтер Петренко
4,S07,вахтер Петренко
5,S09,турист Новак


In [27]:
# Створюємо копії журналів, щоб не змінювати вихідні таблиці
evening_fixed = evening_sql.copy()
backup = backup_sql.copy()

# Використовуємо visit_id як індекс
evening_fixed = evening_fixed.set_index("visit_id")
backup = backup.set_index("visit_id")

# Відновлюємо пропущені назви кімнат із резервного журналу
rooms_fixed = evening_fixed["room"].combine_first(backup["room"])

# Залишаємо лише записи вечірньої зміни
rooms_fixed = rooms_fixed.loc[evening_fixed.index]

# Повертаємо відновлені назви кімнат до таблиці
evening_fixed["room"] = rooms_fixed

# Повертаємо visit_id з індексу до звичайного стовпця
evening_fixed = evening_fixed.reset_index()

# Відновлюємо цілочисельний тип стовпця hour
evening_fixed["hour"] = evening_fixed["hour"].astype(int)

print("Вечірня зміна після відновлення назв кімнат:")
display(evening_fixed)

Вечірня зміна після відновлення назв кімнат:


,visit_id,suspect_id,room,hour,sensor_id
0,V002,S02,Бароко,15,SEN-518
1,V003,S04,Жовта,18,SEN-457
2,V004,S05,Гобеленова,17,SEN-646
3,V006,S05,Антична,15,SEN-351
4,V009,S01,Єгипетська,20,SEN-626
5,V010,S07,Єгипетська,18,SEN-559
6,V012,S06,Антична,15,SEN-710
7,V013,S05,Вітражна,14,SEN-158
8,V014,S07,Єгипетська,17,SEN-599
9,V015,S10,Єгипетська,15,SEN-484


In [28]:
# Об'єднуємо вечірній і ранковий журнали
visits_all = pd.concat(
    [evening_fixed, morning_sql],
    ignore_index=True
)

# Видаляємо службовий стовпець sensor_id
visits_all = visits_all.drop(columns=["sensor_id"])

# Повертаємо visit_id до звичайного стовпця, якщо це необхідно
if "visit_id" not in visits_all.columns:
    visits_all = visits_all.reset_index()

print("Кількість записів в об'єднаному журналі:", len(visits_all))
print("\nОб'єднаний журнал:")
display(visits_all)

Кількість записів в об'єднаному журналі: 52

Об'єднаний журнал:


,visit_id,suspect_id,room,hour
0,V002,S02,Бароко,15
1,V003,S04,Жовта,18
2,V004,S05,Гобеленова,17
3,V006,S05,Антична,15
4,V009,S01,Єгипетська,20
5,V010,S07,Єгипетська,18
6,V012,S06,Антична,15
7,V013,S05,Вітражна,14
8,V014,S07,Єгипетська,17
9,V015,S10,Єгипетська,15


In [29]:
# Додаємо дані про працівників до журналу відвідувань
visits_named = visits_all.merge(
    suspects_sql,
    on="suspect_id",
    how="left"
)

print("Журнал із даними про працівників:")
display(visits_named)

Журнал із даними про працівників:


,visit_id,suspect_id,room,hour,name,role
0,V002,S02,Бароко,15,Захар Тесля,екскурсовод
1,V003,S04,Жовта,18,Марія Снігур,охоронець
2,V004,S05,Гобеленова,17,Ярина Кушнір,науковий співробітник
3,V006,S05,Антична,15,Ярина Кушнір,науковий співробітник
4,V009,S01,Єгипетська,20,Ігор Пасічник,реставратор
5,V010,S07,Єгипетська,18,Раїса Дубова,фотограф
6,V012,S06,Антична,15,Устим Романко,прибиральник
7,V013,S05,Вітражна,14,Ярина Кушнір,науковий співробітник
8,V014,S07,Єгипетська,17,Раїса Дубова,фотограф
9,V015,S10,Єгипетська,15,Андрій Лозовий,екскурсовод


In [30]:
# Порівнюємо чотири типи об'єднання свідчень із довідником підозрюваних

merge_inner = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="inner"
)

merge_left = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="left"
)

merge_right = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="right"
)

merge_outer = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="outer"
)

print("Кількість рядків при inner:", len(merge_inner))
print("Кількість рядків при left:", len(merge_left))
print("Кількість рядків при right:", len(merge_right))
print("Кількість рядків при outer:", len(merge_outer))

print("\nПриклад результату outer:")
display(merge_outer)

Кількість рядків при inner: 7
Кількість рядків при left: 11
Кількість рядків при right: 12
Кількість рядків при outer: 16

Приклад результату outer:


,witness,saw_suspect_id,room,hour,suspect_id,name,role
0,NaN,NaN,NaN,NaN,S01,Ігор Пасічник,реставратор
1,вахтер Петренко,S02,Жовта,9.0,S02,Захар Тесля,екскурсовод
2,NaN,NaN,NaN,NaN,S03,Галина Мірошник,електрик
3,вахтер Петренко,S04,Гобеленова,13.0,S04,Марія Снігур,охоронець
4,волонтер Сидір,S04,Вітражна,20.0,S04,Марія Снігур,охоронець
5,NaN,NaN,NaN,NaN,S05,Ярина Кушнір,науковий співробітник
6,водій Юрченко,S06,Жовта,19.0,S06,Устим Романко,прибиральник
7,NaN,NaN,NaN,NaN,S07,Раїса Дубова,фотограф
8,вахтер Петренко,S08,Єгипетська,12.0,S08,Еліна Бортник,доглядач
9,NaN,NaN,NaN,NaN,S09,Павло Чумак,реставратор


In [31]:
try:
    pivot_result = visits_named.pivot(
        index="name",
        columns="room",
        values="visit_id"
    )
    display(pivot_result)
except ValueError as e:
    print("Помилка pivot:", e)

Помилка pivot: Index contains duplicate entries, cannot reshape


In [32]:
pivot_table_result = visits_named.pivot_table(
    index="name",
    columns="room",
    values="visit_id",
    aggfunc="count",
    fill_value=0
)

print("Кількість відвідувань кожної кімнати працівниками:")
display(pivot_table_result)

Кількість відвідувань кожної кімнати працівниками:


room,Єгипетська,Антична,Бароко,Вітражна,Гобеленова,Дзеркальна,Жовта,Зброярня
name,,,,,,,,
Ігор Пасічник,1,0,1,1,1,0,0,0
Андрій Лозовий,1,1,0,2,0,0,0,1
Галина Мірошник,0,2,0,0,0,1,2,0
Еліна Бортник,1,0,0,0,2,0,0,0
Захар Тесля,1,0,1,1,1,1,1,0
Марія Снігур,1,0,0,1,1,0,1,0
Павло Чумак,1,3,0,0,0,0,1,0
Раїса Дубова,3,3,0,0,0,0,0,1
Устим Романко,0,1,0,0,1,0,2,2


In [33]:
with open(CASE_DIR / "case.json", encoding="utf-8") as f:
    case_data = json.load(f)

print("Умови справи:", case_data)

Умови справи: {'n': 8, 'crime_room': 'Антична', 'crime_hour': 15}


In [34]:
crime_room = case_data["crime_room"]
crime_hour = case_data["crime_hour"]

suspects_at_crime = visits_named[
    (visits_named["room"] == crime_room) &
    (visits_named["hour"] == crime_hour)
]

print("Підозрювані, які відвідували кімнату «Антична» о 15:00:")
display(suspects_at_crime[["visit_id", "suspect_id", "name", "room", "hour"]])

Підозрювані, які відвідували кімнату «Антична» о 15:00:


,visit_id,suspect_id,name,room,hour
3,V006,S05,Ярина Кушнір,Антична,15
6,V012,S06,Устим Романко,Антична,15
19,V046,S07,Раїса Дубова,Антична,15


In [35]:
alibi_ids = set(alibi["suspect_id"])

suspects_at_crime = suspects_at_crime.copy()
suspects_at_crime["has_alibi"] = suspects_at_crime["suspect_id"].isin(alibi_ids)

print("Перевірка алібі підозрюваних:")
display(suspects_at_crime[["suspect_id", "name", "has_alibi"]])

Перевірка алібі підозрюваних:


,suspect_id,name,has_alibi
3,S05,Ярина Кушнір,True
6,S06,Устим Романко,False
19,S07,Раїса Дубова,True


In [36]:
# Визначаємо підозрюваного, який не має алібі
guilty = suspects_at_crime[
    ~suspects_at_crime["has_alibi"]
].merge(
    suspects_sql,
    on="suspect_id",
    how="left",
    suffixes=("", "_suspect")
)

print("Винний у справі:")
display(guilty[["suspect_id", "name_suspect", "role"]].rename(
    columns={"name_suspect": "name"}
))

Винний у справі:


,suspect_id,name,role
0,S06,Устим Романко,прибиральник
